# Full Model CPT - 8-Bit with Custom Training Loop
## Qwen2.5-7B - No Trainer API Restrictions

- Custom training loop (bypasses Trainer validation)
- 8-bit quantized model
- NO adapters, NO LoRA
- Full parameter training
- Fits on 31.84GB GPU

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'

import torch
import json
from pathlib import Path
from datetime import datetime
from typing import Dict, List
import warnings
warnings.filterwarnings('ignore')

print("\n" + "="*80)
print("FULL MODEL CPT - 8-BIT WITH CUSTOM TRAINING LOOP")
print("="*80)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("="*80 + "\n")

In [ ]:
# ============ CONFIG ============
config = {
    "train_file": "pretraining_augmented_data/train.jsonl",
    "eval_file": "pretraining_augmented_data/eval.jsonl",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-5,
    "warmup_steps": 100,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "max_seq_length": 256,
    "output_dir": "medical_qwen_cpt_8bit_custom",
    "save_steps": 50,
    "eval_steps": 25,
    "logging_steps": 5,
    "seed": 42,
}

print("Configuration:")
print("="*70)
print(f"Model: Qwen2.5-7B (8-bit, full params, no LoRA)")
print(f"Batch size: {config['per_device_train_batch_size']}")
print(f"Gradient accumulation: {config['gradient_accumulation_steps']}")
print(f"Effective batch: {config['per_device_train_batch_size'] * config['gradient_accumulation_steps']}")
print(f"Sequence length: {config['max_seq_length']}")
print(f"Learning rate: {config['learning_rate']}")
print("="*70 + "\n")

In [ ]:
# ============ LOAD DATA ============
def load_jsonl(file_path: str) -> List[Dict]:
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            try:
                data.append(json.loads(line))
            except:
                pass
    return data

print("Loading data...")
train_data = load_jsonl(config["train_file"])
eval_data = load_jsonl(config["eval_file"])

print(f"✅ Train: {len(train_data):,} chunks")
print(f"✅ Eval:  {len(eval_data):,} chunks\n")

In [ ]:
# ============ LOAD TOKENIZER ============
from transformers import AutoTokenizer

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen2.5-7B",
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Tokenizer loaded\n")

In [ ]:
# ============ LOAD MODEL IN 8-BIT ============
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

print("Loading model in 8-bit...")

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16,
    bnb_8bit_use_double_quant=True,
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model = AutoModelForCausalLM.from_pretrained(
    "Qwen2.5-7B",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

print(f"✅ Model loaded")
num_params = sum(p.numel() for p in model.parameters())
print(f"   Parameters: {num_params/1e9:.2f}B")
print(f"   Quantization: 8-bit")
print(f"   Gradient checkpointing: ENABLED")

allocated = torch.cuda.memory_allocated(0) / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
free = total - allocated

print(f"\n   GPU allocated: {allocated:.2f} / {total:.2f} GB")
print(f"   GPU free: {free:.2f} GB\n")

In [ ]:
# ============ TOKENIZE & PREPARE DATASETS ============
from datasets import Dataset
from torch.utils.data import DataLoader

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=config["max_seq_length"],
        padding="max_length",
    )

print("Tokenizing datasets...")
train_dataset = Dataset.from_dict({"text": [c["text"] for c in train_data]})
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["text"],
)

eval_dataset = Dataset.from_dict({"text": [c["text"] for c in eval_data]})
eval_dataset = eval_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["text"],
)

# Create DataLoaders
def collate_fn(batch):
    return {
        'input_ids': torch.stack([torch.tensor(x['input_ids']) for x in batch]),
        'attention_mask': torch.stack([torch.tensor(x['attention_mask']) for x in batch]),
    }

train_loader = DataLoader(
    train_dataset,
    batch_size=config["per_device_train_batch_size"],
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=False,
    num_workers=0,
)

eval_loader = DataLoader(
    eval_dataset,
    batch_size=config["per_device_eval_batch_size"],
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=False,
    num_workers=0,
)

print(f"✅ Train: {len(train_dataset):,} samples")
print(f"✅ Eval: {len(eval_dataset):,} samples\n")

In [ ]:
# ============ SETUP OPTIMIZER & SCHEDULER ============
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR
from bitsandbytes.optim import AdamW8bit

optimizer = AdamW8bit(
    model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)

total_steps = len(train_loader) * config["num_train_epochs"] // config["gradient_accumulation_steps"]
scheduler = LinearLR(
    optimizer,
    start_factor=1.0,
    total_iters=total_steps,
)

print(f"✅ Optimizer: AdamW8bit (8-bit optimizer state)")
print(f"✅ Scheduler: Linear warmup")
print(f"   Total steps: {total_steps}\n")

In [ ]:
# ============ TRAINING LOOP ============

output_dir = Path(config["output_dir"])
output_dir.mkdir(exist_ok=True)

model.train()
global_step = 0
epoch_loss = 0
step_count = 0

print("="*80)
print("🚀 STARTING TRAINING")
print("="*80)
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

for epoch in range(config["num_train_epochs"]):
    print(f"\nEpoch {epoch + 1}/{config['num_train_epochs']}")
    print("-" * 70)
    
    epoch_loss = 0
    step_count = 0
    
    for batch_idx, batch in enumerate(train_loader):
        # Move batch to device
        batch = {k: v.to(model.device) for k, v in batch.items()}
        
        # Forward pass
        outputs = model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['input_ids'],
        )
        
        loss = outputs.loss
        
        # Scale loss for gradient accumulation
        loss = loss / config["gradient_accumulation_steps"]
        
        # Backward pass
        loss.backward()
        
        epoch_loss += loss.item() * config["gradient_accumulation_steps"]
        step_count += 1
        
        # Gradient accumulation step
        if (batch_idx + 1) % config["gradient_accumulation_steps"] == 0:
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
            
            # Optimizer step
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            global_step += 1
            
            # Logging
            if global_step % config["logging_steps"] == 0:
                avg_loss = epoch_loss / step_count
                print(f"Step {global_step}: loss = {avg_loss:.4f}")
            
            # Save checkpoint
            if global_step % config["save_steps"] == 0:
                checkpoint_dir = output_dir / f"checkpoint-{global_step}"
                checkpoint_dir.mkdir(exist_ok=True)
                model.save_pretrained(str(checkpoint_dir))
                tokenizer.save_pretrained(str(checkpoint_dir))
                print(f"✓ Checkpoint saved: {checkpoint_dir}")
            
            # Evaluation
            if global_step % config["eval_steps"] == 0:
                model.eval()
                eval_loss = 0
                eval_steps = 0
                
                with torch.no_grad():
                    for eval_batch in eval_loader:
                        eval_batch = {k: v.to(model.device) for k, v in eval_batch.items()}
                        eval_outputs = model(
                            input_ids=eval_batch['input_ids'],
                            attention_mask=eval_batch['attention_mask'],
                            labels=eval_batch['input_ids'],
                        )
                        eval_loss += eval_outputs.loss.item()
                        eval_steps += 1
                
                avg_eval_loss = eval_loss / eval_steps
                print(f"  Eval loss: {avg_eval_loss:.4f}")
                model.train()
    
    epoch_avg_loss = epoch_loss / step_count
    print(f"Epoch {epoch + 1} - Avg loss: {epoch_avg_loss:.4f}")

print("\n" + "="*80)
print("✅ TRAINING COMPLETE")
print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

In [ ]:
# ============ SAVE FINAL MODEL ============
best_model_path = output_dir / "best_model"
best_model_path.mkdir(exist_ok=True)

print(f"\nSaving final model to {best_model_path}...")
model.save_pretrained(str(best_model_path))
tokenizer.save_pretrained(str(best_model_path))

print(f"✅ Model saved")
print(f"\nTraining outputs in: {output_dir}")